# Simple MNIST convnet via PyTorch

**Adapted from:** the Keras example [Simple MNIST convnet](https://keras.io/examples/vision/mnist_convnet/) by [fchollet](https://twitter.com/fchollet), rewritten in **PyTorch**.<br>
**Description:** A simple convnet that achieves ~99% test accuracy on MNIST.

> In Colab, a GPU (**Runtime → Change runtime type → GPU**) makes training faster, but this small model also trains in a few minutes on a CPU.

## Setup

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():   # Apple-silicon Macs
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

## Prepare the data

`torchvision` downloads MNIST for us. `ToTensor()` turns each 28×28 image into a float tensor of shape `(1, 28, 28)` scaled to the [0, 1] range. PyTorch puts the **channel dimension first** (`N, C, H, W`), whereas Keras puts it last (`N, H, W, C`).

Labels stay as integer class ids (0–9). There is no need to one-hot encode them, because `nn.CrossEntropyLoss` takes class indices directly.

In [ ]:
# Model / data parameters
num_classes = 10
input_shape = (1, 28, 28)

transform = transforms.ToTensor()
train_full = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="data", train=False, download=True, transform=transform)

# Hold out 10% of the training set for validation (like Keras' validation_split=0.1)
n_val = len(train_full) // 10
train_ds, val_ds = random_split(train_full, [len(train_full) - n_val, n_val])

x, y = train_full[0]
print("one image shape:", tuple(x.shape), " label:", y, " pixel range:", (x.min().item(), x.max().item()))
print(len(train_ds), "train samples")
print(len(val_ds), "validation samples")
print(len(test_ds), "test samples")

In [ ]:
batch_size = 128

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)
test_loader = DataLoader(test_ds, batch_size=batch_size)

xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), tuple(yb.shape))

## Build the model

The same architecture as the Keras version:

`Conv(32, 3×3) → ReLU → MaxPool(2×2) → Conv(64, 3×3) → ReLU → MaxPool(2×2) → Flatten → Dropout(0.5) → Linear(10)`

Shape trace: `1×28×28 → 32×26×26 → 32×13×13 → 64×11×11 → 64×5×5 → 1600 → 10`.

Note that the last layer outputs raw scores (**logits**) with no softmax: `nn.CrossEntropyLoss` applies log-softmax internally, which is more numerically stable.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(32, 64, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Dropout(0.5),
    nn.Linear(64 * 5 * 5, num_classes),
).to(device)

print(model)

# Print the output shape of each layer, like Keras' model.summary()
out = torch.zeros(1, *input_shape, device=device)
for layer in model:
    out = layer(out)
    print(f"{layer.__class__.__name__:10s} -> {tuple(out.shape)}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

## Train the model

Keras' `compile` + `fit` hide the training loop. In PyTorch we write it ourselves: for each batch, run a forward pass, compute the loss, back-propagate (`loss.backward()`), and take an optimizer step. `model.train()` / `model.eval()` switch dropout on and off.

In [ ]:
epochs = 5

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())   # default lr=1e-3, same as Keras' "adam"


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += len(yb)
    return total_loss / n, correct / n


for epoch in range(epochs):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += len(yb)

    val_loss, val_acc = evaluate(val_loader)
    print(f"Epoch {epoch + 1}/{epochs} - loss: {total_loss / n:.4f} - accuracy: {correct / n:.4f}"
          f" - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

## Evaluate the trained model

In [ ]:
test_loss, test_acc = evaluate(test_loader)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

## Look at some predictions

Applying `softmax` to the logits turns them into class probabilities.

In [ ]:
import matplotlib.pyplot as plt

model.eval()
xb, yb = next(iter(test_loader))
with torch.no_grad():
    probs = torch.softmax(model(xb.to(device)), dim=1).cpu()
preds = probs.argmax(dim=1)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(xb[i, 0], cmap="gray")
    color = "green" if preds[i] == yb[i] else "red"
    ax.set_title(f"pred {preds[i].item()} ({probs[i].max():.2f})", color=color, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()